# 🎬 مسجّل الدورة — Google Colab (بدون جهازك، كوكيز دائمة)

يشتغل على سحابة Google (مش جهازك، ما يستهلك مساحة)، وينزّل بث/فيديو يوتيوب ثم
يرفعه لتيليجرام (رسائلك المحفوظة). الكوكيز تُحفظ على Google Drive وتتجدّد لحالها
كل تشغيل، فتبقى حيّة بدون ما تجدّدها كل يوم.

## الإعداد مرة واحدة
1. **حساب Google احتياطي** (مش حسابك الشخصي): سجّل فيه دخول يوتيوب، صدّر كوكيزه
   (إضافة متصفح «Get cookies.txt LOCALLY») من نافذة Incognito ثم سكّرها بدون تسجيل خروج.
2. ارفع ملف الكوكيز على Drive باسم **`yt_cookies.txt`** في **My Drive** (الجذر).
3. بالقائمة اليسار بـ Colab: أيقونة المفتاح 🔑 (Secrets) → أضف:
   - `API_ID` , `API_HASH` , `SESSION_STRING` (نفس اللي عندك)، و`TARGET` اختياري.
   - فعّل «Notebook access» لكل واحد.

## كل يوم
شغّل الخلايا بالترتيب (أو Runtime → Run all)، وبخلية «التنزيل» حطّ رابط البث.

### 1) تثبيت الأدوات (yt-dlp + EJS + deno + أدوات الرفع)

In [ ]:
!pip -q install -U "yt-dlp[default]" yt-dlp-ejs kurigram tgcrypto
# deno = JS runtime ليوتيوب (من إصدارات GitHub، بدون حساب)
!curl -fsSL -o /tmp/deno.zip https://github.com/denoland/deno/releases/latest/download/deno-x86_64-unknown-linux-gnu.zip
!unzip -o -q /tmp/deno.zip -d /usr/local/bin && chmod +x /usr/local/bin/deno
import shutil
print("deno:", shutil.which("deno"))
print("ffmpeg:", shutil.which("ffmpeg"))  # مثبّت مسبقًا على Colab
!yt-dlp --version

### 2) ربط Google Drive (لحفظ الكوكيز وتجديدها لحالها)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
COOKIES = '/content/drive/MyDrive/yt_cookies.txt'
print("ملف الكوكيز موجود؟", os.path.exists(COOKIES))
if not os.path.exists(COOKIES):
    print("⚠️ ارفع yt_cookies.txt إلى My Drive (الجذر) أول مرة.")

### 3) قراءة بيانات تيليجرام من Secrets 🔑

In [ ]:
from google.colab import userdata
API_ID = int(userdata.get('API_ID'))
API_HASH = userdata.get('API_HASH')
SESSION_STRING = userdata.get('SESSION_STRING')
try:
    TARGET = userdata.get('TARGET') or 'me'
except Exception:
    TARGET = 'me'
print("جاهز. الهدف:", TARGET)

### 4) التنزيل — حطّ رابط البث هنا 👇
الكوكيز تُقرأ من Drive وتُحدَّث مكانها تلقائيًا (تجديد دائم).

In [ ]:
URL = ""  #@param {type:"string"}
QUALITY = "best"  #@param ["best", "1080", "720", "480"]

import os, glob
os.makedirs('/content/out', exist_ok=True)
for f in glob.glob('/content/out/*'):
    os.remove(f)

fmt = "bv*+ba/b" if QUALITY == "best" else f"bv*[height<={QUALITY}]+ba/b[height<={QUALITY}]/b"
ck = f'--cookies "{COOKIES}"' if os.path.exists(COOKIES) else ''

# --cookies على ملف Drive => yt-dlp يحفظ الكوكيز المحدّثة مكانها = تجديد دائم.
!yt-dlp -v --live-from-start {ck} -f "{fmt}" \
  --js-runtimes deno \
  --extractor-args "youtube:player_client=tv,mweb,web_safari,default,web" \
  --merge-output-format mp4 --no-part \
  -P /content/out -o "%(title)s [%(id)s].%(ext)s" \
  "{URL}"

print("\n--- الملفات ---")
!ls -lh /content/out

### 5) الرفع لتيليجرام (حتى 4GB بحساب Premium)

In [ ]:
import glob, os, asyncio
from pyrogram import Client

files = sorted(f for f in glob.glob('/content/out/*') if os.path.isfile(f))
assert files, "❌ لا يوجد ملف للرفع (فشل التنزيل؟ شوف لوج الخلية 4)"
target = int(TARGET) if str(TARGET).lstrip('-').isdigit() else TARGET

async def up():
    app = Client("up", api_id=API_ID, api_hash=API_HASH, session_string=SESSION_STRING, in_memory=True)
    await app.start()
    me = await app.get_me()
    print("دخلت كـ:", me.first_name, "| premium=", getattr(me, "is_premium", False))
    last = {"p": -1}
    def prog(c, t):
        p = int(c*100/t) if t else 0
        if p != last["p"] and p % 10 == 0:
            last["p"] = p; print(f"  ⬆️ {p}%")
    for f in files:
        print(f"⬆️ {os.path.basename(f)} ({os.path.getsize(f)/1e9:.2f} GB)")
        last["p"] = -1
        await app.send_document(target, f, caption=os.path.basename(f), force_document=True, progress=prog)
        print("✅ تم")
    await app.stop()

await up()
print("🎉 خلص — شوف رسائلك المحفوظة بتيليجرام.")